In [1]:
!cp -r /kaggle/input/code-focus/FOCUS/ /kaggle/working/
import os
!cd /kaggle/working

In [2]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -y

!pip uninstall -y pyspark
!pip install pyspark==3.5.1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



The following additional packages will be installed:
  openjdk-11-jre-headless
Suggested packages:
  openjdk-11-demo openjdk-11-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following packages will be upgraded:
  openjdk-11-jdk-headless openjdk-11-jre-headless
2 upgraded, 0 newly installed, 0 to remove and 176 not upgraded.
Need to get 116 MB of archives.
After this operation, 89.1 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 openjdk-11-jdk-headless amd64 11.0.28+6-1ubuntu1~22.04.1 [73.6 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 openjdk-11-jre-headless amd64 11.0.28+6-1ubuntu1~22.04.1 [42.6 MB]
Fetched 116 MB in 6s (18.0 

In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkTestNew") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/26 22:23:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [4]:
#Thêm cột case_id và sile_id: case_id: mã người, silde_id: mã ảnh (data không có)
import pandas as pd
import os
csv_src = "/kaggle/input/ubc-ocean/UBC-OCEAN/train.csv"
csv_fixed = "/kaggle/working/FOCUS/train.csv"
if not os.path.exists(csv_fixed):  
    df = pd.read_csv(csv_src)
    df["case_id"] = df["image_id"].astype(str)
    df["slide_id"] = df["image_id"].astype(str)
    df.to_csv(csv_fixed, index=False)
print("✅ CSV fixed saved at:", csv_fixed)

✅ CSV fixed saved at: /kaggle/working/FOCUS/train.csv


In [5]:
#Bắt buộc

!mkdir -p /kaggle/working/FOCUS/ckpts
!mkdir -p /kaggle/working/FOCUS/features
# !mkdir -p /kaggle/working/FOCUS/features/features.csv

!cp /kaggle/input/conch-ckpts/conch.pth /kaggle/working/FOCUS/ckpts/conch.pth


In [6]:
# #Trích xuất đặc trưng ảnh
!cp -rn /kaggle/input/code-focus/FOCUS/ /kaggle/working/
project_dir = '/kaggle/working/FOCUS'
os.chdir(project_dir)
!mkdir -p results/FOCUS/conch/

# !spark-submit \
#   --master local[*] \
#   --driver-memory 8g \
#   --executor-memory 6g \
#   /kaggle/working/FOCUS/map_reduce_features.py \
#   --csv_path /kaggle/working/FOCUS/train.csv \
#   --source_folder /kaggle/input/ubc-ocean/UBC-OCEAN/train_thumbnails \
#   --output_csv /kaggle/working/FOCUS/features/features.csv \
#   --features_dir /kaggle/working/FOCUS/features \
#   --ckpt_path /kaggle/working/FOCUS/ckpts/conch.pth \
#   2>&1 | tee spark_output.log


In [7]:
import os
import subprocess
from joblib import Parallel, delayed
import glob
import pandas as pd
import torch

os.chdir('/kaggle/working/FOCUS/')
!mkdir -p results/FOCUS/conch/KAVTC

# Cài thêm dependencies
!pip install --quiet tensorboardX openai-clip faiss-cpu

CORRECT_CSV_PATH = "/kaggle/working/FOCUS/train.csv"

# Update đường dẫn csv trong main.py
!sed -i -e "s|csv_path = '.*'|csv_path = '{CORRECT_CSV_PATH}'|g" main.py
print(f"✅ Đã cập nhật đường dẫn trong main.py thành: {CORRECT_CSV_PATH}")


def run_fold(fold_id, gpu_id, total_folds, shot):
    import sys
    exp_code = f"UBC-OCEAN_{shot}shots_{total_folds}folds"
    split_dir = f"UBC-OCEAN_{shot}shots_{total_folds}folds"

    log_dir = f"/kaggle/working/FOCUS/results/FOCUS/conch/{exp_code}"
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f"fold_{fold_id}.log")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    cmd = [
        "python", "main.py",
        "--k", str(total_folds),
        "--k_start", str(fold_id),
        "--k_end", str(fold_id + 1),
        "--seed", "1",
        "--drop_out",
        "--early_stopping",
        "--lr", "1e-4",
        "--label_frac", "1",
        "--bag_loss", "ce",
        "--task", "task_UBC-OCEAN_subtyping",
        "--results_dir", "results/FOCUS/conch/KAVTC",
        "--exp_code", exp_code,
        "--model_type", "FOCUS",
        "--mode", "transformer",
        "--max_epochs", "100",
        "--log_data",
        "--data_root_dir", "/kaggle/input/ubc-ovarian-cancer-subtype-classification/",
        "--data_folder_s", "/kaggle/input/feature-data/features/features/",
        "--data_folder_l", "/kaggle/input/feature-data/features/features/",
        "--split_dir", split_dir,
        "--text_prompt_path", "text_prompt/UBC-OCEAN_two_scale_text_prompt.csv",
        "--use_prompt",
        "--use_KAVTC",
        # "--use_SVTC",
        # "--use_CrossAgg",
    ]

    print(f"🚀 Bắt đầu chạy {shot}-shot | Fold {fold_id} trên GPU {gpu_id}...")
    with open(log_file, "w") as f:
        process = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in process.stdout:
            sys.stdout.write(f"[{shot}-shot | Fold {fold_id} | GPU {gpu_id}] {line}")
            sys.stdout.flush()
            f.write(line)
        process.wait()

    print(f"✅ Hoàn thành {shot}-shot | Fold {fold_id}, log lưu tại {log_file}")
    return log_file


def reduce_results(results_dir):
    result_files = glob.glob(os.path.join(results_dir, "result_partial*.csv"))
    print(f"\n🔎 Reduce: Tìm thấy {len(result_files)} file kết quả của các fold trong {results_dir}")

    if len(result_files) == 0:
        print("⚠️ Không tìm thấy file kết quả nào!")
        return None

    df_all = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)
    df_num = df_all.select_dtypes(include=['number'])
    summary = df_num.agg(['mean', 'std']).T
    summary.reset_index(inplace=True)
    summary.rename(columns={'index': 'metric'}, inplace=True)

    output_file = os.path.join(results_dir, "summary.csv")
    summary.to_csv(output_file, index=False)
    print(f"✅ Đã lưu summary vào {output_file}")
    return summary


if __name__ == "__main__":
    num_gpus = torch.cuda.device_count()
    total_folds = 10
    shots = [4, 8, 16]   # Chạy đủ 3 setting

    for shot in shots:
        jobs = [(i, i % num_gpus) for i in range(total_folds)]
        print(f"\n--- MAP stage: {shot}-shot | {len(jobs)} folds trên {num_gpus} GPU ---")
        logs = Parallel(n_jobs=num_gpus)(delayed(run_fold)(fid, gid, total_folds, shot) for fid, gid in jobs)
        print(f"✅ Hoàn thành MAP stage cho {shot}-shot")

        results_dir = f"/kaggle/working/FOCUS/results/FOCUS/conch/KAVTC/UBC-OCEAN_{shot}shots_{total_folds}folds_s1"
        summary_df = reduce_results(results_dir)

        print(f"\n--- KẾT QUẢ {shot}-SHOT ---")
        if summary_df is not None:
            print(summary_df.to_string())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
✅ Đã cập nhật đường dẫn trong main.py thành: /kaggle/working/FOCUS/train.csv

--- MAP stage: 4-shot | 10 folds trên 2 GPU ---
🚀 Bắt đầu chạy 4-shot | Fold 1 trên GPU 1...
[4-shot | Fold 1 | GPU 1] 
[4-shot | Fold 1 | GPU 1] Load Dataset
[4-shot | Fold 1 | GPU 1] label column: label
[4-shot | Fold 1 | GPU 1] label dictionary: {'CC': 0, 'HGSC': 1, 'LGSC': 2, 'EC': 3, 'MC': 4}
[4-shot | Fold 1 | GPU 1] number of classes: 5
[4-shot | Fold 1 | GPU 1] slide-level counts:  
[4-shot | Fold 1 | GPU 1]  label
[4-shot | Fold 1 | GPU 1] 1    222
[4-shot | Fold 1 | GPU 1] 2     47
[4-shot | Fold 1 | GPU 1] 3    124
[4-shot | Fold 1 | GPU 1] 0     99
